In [ ]:
### ESA data download.


import xarray as xr
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
!pip install cartopy
import cartopy.crs as ccrs
import cartopy.mpl.ticker as cticker
from matplotlib import ticker

def wgt_areaave(indat, latS, latN):
    lat = indat.lat
    iplat = lat.where((lat >= latS) & (lat <= latN), drop=True)
    wgt = np.cos(np.deg2rad(lat))
    odat = indat.sel(lat=iplat).weighted(wgt).mean(("lat"), skipna=True)
    return odat

def detrend_dim(da, dim, deg=1):
    p = da.polyfit(dim=dim, deg=deg)
    fit = xr.polyval(da[dim], p.polyfit_coefficients)
    return da - fit

def remove_time_mean(x):
    return x - x.mean(dim='time', skipna=True)

# Define latitude and longitude ranges
min_lat = -90
max_lat = -30
lon_step = 6
lat_step = 6

# Open the dataset
file = f"https://www.oceancolour.org/thredds/dodsC/CCI_ALL-v6.0-MONTHLY?lat[0:{lat_step}:4319],lon[0:{lon_step}:8639],time[0:1:315],chlor_a[0:1:315][0:{lat_step}:4319][0:{lon_step}:8639]"
ds = xr.open_dataset(file)

ds = ds.reindex(lat=ds.lat[::-1])
if ds.lat[0] > ds.lat[-1]:
    ds = ds.reindex(lat=ds.lat[::-1])

# 2️⃣ Now safely select the Southern Ocean band
ds1 = ds.sel(lat=slice(-90, -30))

# 3️⃣ Verify the result
print(ds1.lat.min().item(), ds1.lat.max().item())
print(ds1.dims)

outfile = "ESA_Chl_south_surface.nc"
ds1.to_netcdf(outfile)

In [ ]:
# BSOSE Chl data download

import os
import xarray as xr
url = "https://sose.ucsd.edu/SO6/ITER155/MONTHLY/Chl_bsoseI155_2013to2023_monthly.nc"
local_path = "Chl_bsoseI155_2013to2023_monthly.nc"

# Download if not present
if not os.path.exists(local_path):
    print(f"Downloading {local_path} ...")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(local_path, "wb") as f:
            for chunk in tqdm(r.iter_content(chunk_size=8192)):
                f.write(chunk)

# Open locally
data = xr.open_dataset(local_path, engine="netcdf4")


In [ ]:
# Eppley VGPM data download

#!pip install pyhdf
import os
import requests
import gzip
import shutil
from datetime import datetime
import  matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import LogNorm
import matplotlib.ticker as mticker
import numpy as np
from pyhdf.SD import SD, SDC
import numpy as np
import xarray as xr
import pandas as pd
from datetime import datetime

import os, gzip, shutil, time
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def make_session():
    s = requests.Session()
    s.headers.update({"User-Agent": "npp-fetch/1.0"})
    retries = Retry(total=5, backoff_factor=0.5, status_forcelist=[429, 500, 502, 503, 504])
    s.mount("http://", HTTPAdapter(max_retries=retries))
    s.mount("https://", HTTPAdapter(max_retries=retries))
    return s

def download_and_decompress(base_url, date_value, download_dir, session=None):
    if session is None:
        session = make_session()

    gz_filename  = f"vgpm.{date_value}.hdf.gz"
    hdf_filename = f"vgpm.{date_value}.hdf"
    gz_path  = os.path.join(download_dir, gz_filename)
    hdf_path = os.path.join(download_dir, hdf_filename)

    # Skip if already present
    if os.path.exists(hdf_path):
        return hdf_path

    url = f"{base_url}/{gz_filename}"
    # The server often redirects to https; allow that but skip cert verification only for this host.
    try:
        with session.get(url, stream=True, timeout=60, verify=False) as r:  # <-- verify=False
            r.raise_for_status()
            with open(gz_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1<<20):
                    if chunk:
                        f.write(chunk)
        # Decompress
        with gzip.open(gz_path, "rb") as f_in, open(hdf_path, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
        os.remove(gz_path)
        return hdf_path
    except requests.RequestException as e:
        print(f"[skip] {gz_filename}: {e}")
        if os.path.exists(gz_path):
            os.remove(gz_path)
        return None


def read_npp_data(hdf_path):
    try:
        hdf = SD(hdf_path, SDC.READ)
        npp_sds = hdf.select('npp')
        npp_data = npp_sds.get()
        print(f"Extracted NPP data from {hdf_path}")
        return npp_data
    except Exception as e:
        print(f"Error processing {hdf_path}: {e}")
        return None

def main():
    base_url = "http://orca.science.oregonstate.edu/data/1x2/monthly/vgpm.r2022.m.chl.m.sst/hdf"
    download_dir = "./npp_data"
    os.makedirs(download_dir, exist_ok=True)

    # Generate date values for the first day of each month from January 2003 to December 2023
    date_values = []
    for year in range(2003, 2024):
        for month in range(1, 13):
            date = datetime(year, month, 1)
            date_str = date.strftime('%Y%j')
            date_values.append(date_str)

    npp_data_list = []
    for date_value in date_values:
        hdf_path = download_and_decompress(base_url, date_value, download_dir)
        if hdf_path:
            npp_data = read_npp_data(hdf_path)
            if npp_data is not None:
                npp_data_list.append(npp_data)
    return  npp_data_list
    # Further processing of npp_data_list as needed

npp_data_list=main()

from datetime import datetime, timedelta

# Initialize an empty list to store the date strings
date_values = []

# Define the start and end years
start_year = 2003
end_year = 2024

# Loop through each year
for year in range(start_year, end_year + 1):
    # Loop through each day of the year
    for day in range(1, 366):  # 1 to 365
        try:
            # Create a date object for the current year and day
            date = datetime(year, 1, 1) + timedelta(days=day - 1)
            # Format the date as 'YYYYDDD' and append to the list
            date_values.append(date.strftime('%Y%j'))
        except ValueError:
            # This will handle the case for leap years
            continue

# Display the generated date values

# Assuming npp_data_list contains your 252 (lat, lon) arrays
# and date_values contains the corresponding 'YYYYDDD' strings

# Initialize an empty list to store DataArrays
data_arrays = []

# Loop over the NPP data and corresponding date values
for npp_data, date_str in zip(npp_data_list, date_values):
    # Convert 'YYYYDDD' to a datetime object
    date = datetime.strptime(date_str, '%Y%j')
    
    # Create a DataArray for the current NPP data
    da = xr.DataArray(
        npp_data,
        dims=('lat', 'lon'),
        coords={
            'lat': np.linspace(90, -90, npp_data.shape[0]),
            'lon': np.linspace(-180, 180, npp_data.shape[1]),
            'time': date
        },
        name='npp'
    )
    data_arrays.append(da)

# Concatenate all DataArrays along a new 'time' dimension
npp_combined = xr.concat(data_arrays, dim='time')

import xarray as xr

# Assuming 'npp_combined' is your DataArray with dimensions ('time', 'lat', 'lon')

# Reverse the latitude dimension
npp_combined_reversed = npp_combined.reindex(lat=npp_combined.lat[::-1])



# Coarsen the data
npp_coarsened = npp_combined_reversed.coarsen(lat=6, lon=6, boundary='trim').mean()

# Display the resulting DataArray
print(npp_coarsened)


def convert_0_360_to_neg180_180(_ds):
    """
    Convert longitude from 0-360 degrees to -180 to 180 degrees.
    """
    attrs = _ds['lon'].attrs
    if (_ds['lon'].min() >= 0) and (_ds['lon'].max() <= 360):
        with xr.set_options(keep_attrs=True):
            _ds.coords['lon'] = xr.where(_ds['lon'] > 180, _ds['lon'] - 360, _ds['lon'])
        _ds = _ds.sortby('lon')
    return _ds

# Apply the conversion to your dataset
npp_coarsened = convert_0_360_to_neg180_180(npp_coarsened)

# Check latitude order and flip if necessary

# Display the corrected dataset
# Check if latitude is in descending order
if npp_coarsened.lat[0] > npp_coarsened.lat[-1]:
    # Reindex latitude to ascending order
    npp_coarsened = npp_coarsened.reindex(lat=npp_coarsened.lat[::-1])

# Display the corrected dataset
npp_array = npp_coarsened.values

# Replace negative values with NaN
npp_array[npp_array < 0] = np.nan

# Convert back to DataArray
npp_coarsened = xr.DataArray(npp_array, dims=npp_coarsened.dims, coords=npp_coarsened.coords)
npp_coarsened['time']=pd.date_range("2004-01-01", periods=251, freq="M")

npp_coarsened=npp_coarsened.sel(lat=slice(-90,-30))
npp_coarsened=npp_coarsened.isel(time=slice(24,None))


npp_coarsened.name = "npp"

# Optional: add metadata
npp_coarsened.attrs = {
    "long_name": "Net Primary Production (Eppley Algorithm)",
    "units": "mg C m⁻² day⁻¹",
    "description": "Monthly coarsened NPP from Eppley satellite data (2003–2023)",
    "source": "http://orca.science.oregonstate.edu",
    "processing": "Coarsened to 6°×6°, filtered for negative values, longitude converted to -180–180"
}

# Save as NetCDF (compressed)
encoding = {"npp": {"zlib": True, "complevel": 4}}

npp_coarsened.to_netcdf(
    "npp_coarsened.nc",
    format="NETCDF4",
    encoding=encoding
)

npp_coarsened = npp_coarsened.sel(lat=slice(-80, -30))
npp_coarsened = npp_coarsened.sel(
    time=slice("2013-01-31T00:00:00.000000000", "2023-12-31T00:00:00.000000000")
)

mean_npp = npp_coarsened.mean(dim=['lon', 'lat'])

# 3. Plot the time series (e.g. at a given depth level, or integrated)
#    Here we’ll pick the surface depth (Z=0) as an example
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


fig = plt.figure(figsize=(12, 4), dpi=300)
plt.plot(npp_coarsened.time, mean_npp, color='black', linewidth=2)
plt.xlabel("Time", fontsize=18)
plt.ylabel("NPP (mgC/m²/day)", fontsize=18)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.grid(True)
plt.tight_layout()
plt.show()




lon, lat=np.meshgrid(npp_coarsened.lon,npp_coarsened.lat)


for i in range(1):

    fig = plt.figure(figsize=(10, 10), dpi=300)
    ax = plt.subplot(1, 1, 1, projection=ccrs.SouthPolarStereo(central_longitude=0))
    ax.set_extent([-180, 180, -90, -30], crs=ccrs.PlateCarree())

    ax.coastlines(resolution='110m')
    gl = ax.gridlines(draw_labels=True)
    gl.top_labels = False
    gl.right_labels = False
    # Define the yticks for latitude
    #lat_formatter = cticker.LatitudeFormatter()
    #ax.yaxis.set_major_formatter(lat_formatter)

    levels = np.linspace(-600, 600, 40)
    fill = ax.contourf(
    np.array(npp_coarsened.lon),
    np.array(npp_coarsened.lat),
    np.array(npp_coarsened.mean('time')).squeeze(),
    levels=levels,
    cmap=plt.cm.RdBu_r,
    transform=ccrs.PlateCarree()
    )
    # Make the aspect ratio equal to get a circular plot
    ax.set_aspect('equal')

    # Colorbar
    cb = plt.colorbar(fill, orientation='vertical', shrink=0.8)
    font_size = 20  # Adjust as appropriate.
    cb.ax.tick_params(labelsize=font_size)
    cb.set_label('NPP (mgC/m²/day)', fontsize=20, labelpad=15)
    #ax.contour(lon,lat, no3_mean.squeeze(),color='k',cmap=plt.cm.RdBu_r, transform=ccrs.PlateCarree())
    # Set the latitude limits
    ax.set_extent([-180, 180, -90, -30], crs=ccrs.PlateCarree())
    cb.ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f'))
plt.show()




import os
import requests
from tqdm import tqdm
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.path as mpath
import cartopy.mpl.ticker as cticker
from matplotlib import ticker
from scipy.stats import linregress


def linear_trend(y, x):
    """Return slope and p-value of y vs x"""
    mask = np.isfinite(y)
    y_valid = y[mask]
    x_valid = x[mask]

    # Not enough valid points for regression
    if len(y_valid) < 3:
        return np.nan, np.nan

    # All x are identical (no temporal span)
    if np.all(x_valid == x_valid[0]):
        return np.nan, np.nan

    try:
        res = linregress(x_valid, y_valid)
        return res.slope, res.pvalue
    except Exception:
        # Catch any residual numerical corner cases
        return np.nan, np.nan


time_num = npp_coarsened.time.dt.year.values
npp_trend, pval = xr.apply_ufunc(
    linear_trend,
    npp_coarsened,
    time_num,
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[[], []],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float, float]
)
fig = plt.figure(figsize=(10, 10), dpi=300)
ax = plt.subplot(1, 1, 1, projection=ccrs.SouthPolarStereo(central_longitude=0))
ax.set_extent([-180, 180, -90, -30], crs=ccrs.PlateCarree())

ax.coastlines(resolution='110m')
gl = ax.gridlines(draw_labels=True)
gl.top_labels = False
gl.right_labels = False
#lat_formatter = cticker.LatitudeFormatter()
#ax.yaxis.set_major_formatter(lat_formatter)
levels = np.linspace(-15,15, 40)
# Contour plot with PlateCarree projection
fill = ax.contourf(np.array(npp_coarsened.lon), np.array(npp_coarsened.lat), np.array(npp_trend).squeeze(), levels=levels,
cmap=plt.cm.RdBu_r,
transform=ccrs.PlateCarree())
# Make the aspect ratio equal to get a circular plot
ax.set_aspect('equal')
# Colorbar
ax.set_extent([-180, 180, -90, -30], ccrs.PlateCarree())
theta = np.linspace(0, 2*np.pi, 100)
center, radius = [0.5, 0.5], 0.5
verts = np.vstack([np.sin(theta), np.cos(theta)]).T
circle_path = mpath.Path(verts * radius + center)
ax.set_boundary(circle_path, transform=ax.transAxes)
cb = plt.colorbar(fill, orientation='vertical', shrink=0.8)
cb.ax.tick_params(labelsize=20)
cb.set_label('NPP Trend (mgC/m²/day²)', fontsize=20)
#emphasize = ax.contour(lon, lat, np.abs(no3_mean-no3_mean_atlas)  < 1, levels=[0.5], colors='black', linewidths=1.5, transform=ccrs.PlateCarree())
cb.ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f'))
plt.savefig('NPP_trend.pdf')